In [1]:
from pathlib import Path
import os
import pandas as pd

In [2]:
def obtain_task_id(task_id:str):
    return task_id.split('_')[0]

In [ ]:
new_df = pd.DataFrame(columns=[
    'mutation_type',
    'task_id1', 
    'task_id2', 
    'func_input1',
    'model_output1'
    'func_input2',
    'model_output2',
    'reasoning1',
    'reasoning2'
])

csv = Path("/Users/jin/Downloads/new res")
for file_name in os.listdir(csv):
    if not file_name.endswith('.csv'):
        continue

    csv_file = pd.read_csv(csv / file_name)
    task_ids = {obtain_task_id(qid) for qid in csv_file['task_id']}

    for tid in task_ids:
        matching_rows = csv_file['task_id'].str.split('_').str[0] == tid
        subset = csv_file[matching_rows]
        
        fail = []
        correct = []

        for _, ans in subset.iterrows():
            reasoning = ans['reasoning']
            val = ans['failure_type']
            if isinstance(reasoning, float):
                continue
            if isinstance(val, float):  # NaN means passed
                correct.append(ans)
            elif isinstance(val, str) and "AssertionError" in val:
                fail.append(ans)

        if not fail or not correct:
            continue

        for f in fail:
            for c in correct:
                new_df.loc[len(new_df)] = [
                    file_name.split('.csv')[0],
                    f"{f['task_id']} \n AssertionError", 
                    f"{c['task_id']} \n Pass",
                    f['func_input'],
                    f['model_output'],
                    c['func_input'],
                    c['model_output'],
                    f['reasoning'],
                    c['reasoning']
                ]

new_df.to_csv('save.csv', index=False)


ValueError: cannot set a row with mismatched columns